# 🩺 Syght — Healthcare Insurance Fraud Detector Platform
## Unit-1 Experiment 3: Supervised Learning Foundations & Classifier Development
**Course Specialization**: AI & Machine Learning with Cloud & Azure Tools  
**Project**: Healthcare Insurance Fraud Detection ()  

---

### 🎯 Experiment Aim & Overview:
Understand supervised binary classification for healthcare fraud, implement zero-leakage stratified provider partitioning, and evaluate models under class imbalance.

---

### 📚 Theoretical Background
Supervised Learning trains an algorithm on input features $X$ to predict ground truth labels $y \in \{0, 1\}$.

**Crucial Concepts in Fraud Detection**:
1. **Zero Data Leakage**: We must split data at the **Provider level** (Grouped/Stratified split), NEVER at the claim level. If claims from the same provider exist in both train and test sets, the model cheats by memorizing provider IDs!
2. **Why Accuracy Fails (The Accuracy Paradox)**: A trivial model predicting `y=0` achieves 90.4% accuracy on our data while preventing ₹0 in fraud losses.
3. **PR-AUC (Precision-Recall AUC)**: The gold standard metric for imbalanced classification because it evaluates the precision-recall trade-off directly on the rare positive class.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix, precision_recall_curve
)
import matplotlib.pyplot as plt
from ml_pipeline.medicare_simulator import generate_simulated_medicare_data
from ml_pipeline.feature_engineering import build_provider_features
from ml_pipeline.split import zero_leakage_stratified_split
from ml_pipeline.preprocessing import SyghtPreprocessor
from ml_pipeline.config import SyghtConfig

cfg = SyghtConfig()
lbl, bene, ip, op = generate_simulated_medicare_data(n_providers=60, seed=42)
feats = build_provider_features(lbl, bene, ip, op, config=cfg)
train_df, test_df = zero_leakage_stratified_split(feats, test_size=0.20, random_state=42)
print(f'Train Providers: {len(train_df)} | Test Providers: {len(test_df)}')
print(f'Train Fraud Rate: {train_df["fraud"].mean():.2%} | Test Fraud Rate: {test_df["fraud"].mean():.2%}')

### Step 1: Fitting a Baseline Supervised Classifier

In [ ]:
feature_cols = [c for c in feats.columns if c not in ['Provider', 'fraud', 'exposure_amount']]
prep = SyghtPreprocessor(feature_cols).fit(train_df)

X_train = prep.transform(train_df)
y_train = train_df['fraud'].to_numpy()

X_test = prep.transform(test_df)
y_test = test_df['fraud'].to_numpy()

# Fit Logistic Regression with class-weight compensation
clf = LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)
clf.fit(X_train, y_train)

y_prob = clf.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.50).astype(int)
print('Model training complete!')

### Step 2: Comprehensive Evaluation Metric Suite

In [ ]:
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred, zero_division=0)
rec = recall_score(y_test, y_pred, zero_division=0)
f1 = f1_score(y_test, y_pred, zero_division=0)
roc_auc = roc_auc_score(y_test, y_prob)
pr_auc = average_precision_score(y_test, y_prob)

print('--- SUPERVISED EVALUATION METRICS ---')
print(f'Accuracy:         {acc:.4f}')
print(f'Precision:        {prec:.4f}')
print(f'Recall:           {rec:.4f}')
print(f'F1-Score:         {f1:.4f}')
print(f'ROC-AUC:          {roc_auc:.4f}')
print(f'PR-AUC (Primary): {pr_auc:.4f}')

cm = confusion_matrix(y_test, y_pred)
print('\nConfusion Matrix:')
print(cm)

### Step 3: Precision-Recall Curve Plotting

In [ ]:
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_prob)

plt.figure(figsize=(7, 5))
plt.plot(recall_vals, precision_vals, color='#7928ca', lw=2, label=f'LogisticRegression (PR-AUC = {pr_auc:.3f})')
plt.axhline(y=y_test.mean(), color='red', linestyle='--', label=f'Random Baseline ({y_test.mean():.2%})')
plt.xlabel('Recall (Fraud Detection Rate)')
plt.ylabel('Precision (Audit Accuracy)')
plt.title('Precision-Recall Curve on Held-Out Test Set')
plt.legend()
plt.show()